# N=512 Training on Colab (T4 16GB)
Upload the project zip, train with conv trick, download model.

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Upload project zip
from google.colab import files
uploaded = files.upload()  # upload M34Project.zip

In [ ]:
# Step 3: Unzip
!unzip -q M34Project.zip -d /content/project
%cd /content/project

In [ ]:
# Step 4: Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install scipy numpy matplotlib -q

In [ ]:
# Step 5: Verify imports
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Step 6: Train N=512 with conv trick (T4 has 16GB — use big batch)
import torch
import numpy as np
import time
import json
from pathlib import Path
import sys
sys.path.insert(0, '.')

from src.data.poisson import assemble_poisson_2d
from src.models.unet import UNet
from src.training.losses import ConditionLoss
import torch.optim as optim

device = torch.device('cuda')
N = 512
epochs = 300
steps = 100
probes = 64
probe_batch = 64  # T4 has 16GB — can handle big batch
lr = 5e-4
base_features = 32

print(f'N={N} ({N*N:,} DOFs), bf={base_features}, mode=conv')
model = UNet(base_features=base_features, levels=3, dim=2).to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {params:,} params')

# Load previous weights if uploaded
best_path = Path('results/curriculum/2d/N512_bf32/best.pt')
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    print('Loaded previous best.pt')
else:
    print('Training from scratch')

A = assemble_poisson_2d(N)
loss_fn = ConditionLoss(A, N, num_probes=probes, dim=2, mode='conv').to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler('cuda')

save_dir = Path('results/n512_colab')
save_dir.mkdir(parents=True, exist_ok=True)

best_loss = float('inf')
loss_history = []
t_start = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model, device, use_amp=True, probe_batch_size=probe_batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / steps
    loss_history.append(avg_loss)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), save_dir / 'best.pt')
        marker = ' *'
    else:
        marker = ''
    elapsed = time.time() - t_start
    print(f'Epoch {epoch:4d}/{epochs} | loss={avg_loss:.4f} | best={best_loss:.4f} | {elapsed/60:.0f}m{marker}')

torch.save(model.state_dict(), save_dir / 'final.pt')
np.save(save_dir / 'loss_history.npy', np.array(loss_history))
print(f'\nDone. Best loss: {best_loss:.4f}. Time: {(time.time()-t_start)/3600:.1f} hours')

In [ ]:
# Step 7: Test the model
from src.solvers.cg import conjugate_gradient
from src.solvers.fcg import flexible_cg
from src.evaluation.nn_preconditioner import make_nn_preconditioner

model.load_state_dict(torch.load('results/n512_colab/best.pt', map_location=device, weights_only=True))
nn = make_nn_preconditioner(model, N, device=device, dim=2)

rng = np.random.default_rng(99)
b = rng.standard_normal(N*N)
res_cg = conjugate_gradient(A, b, tol=1e-6)
res_nn = flexible_cg(A, b, nn, tol=1e-6, max_iter=5000, m_max=20)
red = 1 - res_nn.iterations / res_cg.iterations
print(f'CG: {res_cg.iterations} iters')
print(f'NN+FCG: {res_nn.iterations} iters ({red:+.1%}), converged={res_nn.converged}')

In [ ]:
# Step 8: Download trained model
from google.colab import files
files.download('results/n512_colab/best.pt')
files.download('results/n512_colab/final.pt')